In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, explode, 
    from_unixtime, round
)
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    FloatType, LongType, IntegerType
)
import requests
import json

# Storage config
storage_account = "theportfoliostorage"
container = "datalake"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-id"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", 
               f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='de-portfolio-scope', key='sp-tenant-id')}/oauth2/token")

print("Storage configured successfully")

In [0]:
# Hit the CoinGecko API and see what comes back
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False
}

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")
print(f"Number of coins returned: {len(response.json())}")
print("\nFirst coin raw data:")
print(json.dumps(response.json()[0], indent=2))

In [0]:
# Define schema explicitly - never trust inferSchema for streaming
schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("current_price", FloatType(), True),
    StructField("market_cap", LongType(), True),
    StructField("market_cap_rank", IntegerType(), True),
    StructField("total_volume", LongType(), True),
    StructField("high_24h", FloatType(), True),
    StructField("low_24h", FloatType(), True),
    StructField("price_change_24h", FloatType(), True),
    StructField("price_change_percentage_24h", FloatType(), True),
    StructField("circulating_supply", FloatType(), True),
    StructField("total_supply", FloatType(), True),
    StructField("max_supply", FloatType(), True),
    StructField("last_updated", StringType(), True)
])

# Parse API response into a DataFrame
data = response.json()

# Extract only the fields we need
rows = []
for coin in data:
    rows.append((
        coin.get("id"),
        coin.get("symbol"),
        coin.get("name"),
        coin.get("current_price"),
        coin.get("market_cap"),
        coin.get("market_cap_rank"),
        coin.get("total_volume"),
        coin.get("high_24h"),
        coin.get("low_24h"),
        coin.get("price_change_24h"),
        coin.get("price_change_percentage_24h"),
        coin.get("circulating_supply"),
        coin.get("total_supply"),
        coin.get("max_supply"),
        coin.get("last_updated")
    ))

df_raw = spark.createDataFrame(rows, schema)

# Add ingestion metadata
df_raw = df_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_batch_id", lit(1))

print(f"Rows in this batch: {df_raw.count()}")
df_raw.show(truncate=False)

In [0]:
# Bronze path
bronze_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/raw/crypto/prices/"

# Function that fetches one batch from the API
def fetch_crypto_batch(batch_id):
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 10,
        "page": 1,
        "sparkline": False
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    rows = []
    for coin in data:
        rows.append((
            coin.get("id"),
            coin.get("symbol"),
            coin.get("name"),
            coin.get("current_price"),
            coin.get("market_cap"),
            coin.get("market_cap_rank"),
            coin.get("total_volume"),
            coin.get("high_24h"),
            coin.get("low_24h"),
            coin.get("price_change_24h"),
            coin.get("price_change_percentage_24h"),
            coin.get("circulating_supply"),
            coin.get("total_supply"),
            coin.get("max_supply"),
            coin.get("last_updated")
        ))
    
    df = spark.createDataFrame(rows, schema)
    df = df \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_batch_id", lit(batch_id))
    
    return df

# Test the function works
test_df = fetch_crypto_batch(1)
print(f"Batch test successful - {test_df.count()} rows fetched")

In [0]:
import time

# Production version - one batch per scheduled run
# Databricks Workflow calls this every 30 minutes
batch_id = int(time.time())

print(f"Starting batch {batch_id}")

df_batch = fetch_crypto_batch(batch_id)

df_batch.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("crypto.bronze.prices_raw")

total_rows = spark.sql(
    "SELECT COUNT(*) as total FROM crypto.bronze.prices_raw"
).collect()[0][0]

print(f"Batch {batch_id} written successfully")
print(f"Total rows in Bronze table: {total_rows}")

In [0]:
spark.sql("SELECT COUNT(*) as total_rows FROM crypto.bronze.prices_raw").show()

In [0]:
spark.sql("""
    SELECT 
        name,
        current_price,
        price_change_percentage_24h,
        _ingested_at,
        _batch_id
    FROM crypto.bronze.prices_raw
    WHERE symbol = 'btc'
    ORDER BY _batch_id
""").show(truncate=False)

In [0]:
spark.sql("SELECT COUNT(*) as total FROM crypto.bronze.prices_raw").show()

In [0]:
spark.sql("""
    SELECT _batch_id, COUNT(*) as rows, MIN(_ingested_at) as ingested_at
    FROM crypto.bronze.prices_raw
    GROUP BY _batch_id
    ORDER BY ingested_at DESC
    LIMIT 5
""").show()